In [2]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, f1_score, recall_score
)
from sklearn.preprocessing import label_binarize
from matplotlib.backends.backend_pdf import PdfPages

# Rutas
INPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\selected"
OUTPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\models"
os.makedirs(OUTPUT_PATH, exist_ok=True)

metricas_totales = []
mejor_modelo = None
mejor_score = -1

# Identificar todos los conjuntos
archivos_train = [f for f in os.listdir(INPUT_PATH) if f.startswith("X_train_")]

for train_file in archivos_train:
    nombre_base = train_file.replace("X_train_", "").replace(".parquet", "")
    try:
        print(f"\n🔍 Entrenando modelo: {nombre_base}")

        # Cargar datasets
        X_train = pd.read_parquet(os.path.join(INPUT_PATH, f"X_train_{nombre_base}.parquet"))
        X_test = pd.read_parquet(os.path.join(INPUT_PATH, f"X_test_{nombre_base}.parquet"))
        y_train = pd.read_parquet(os.path.join(INPUT_PATH, f"y_train_{nombre_base}.parquet")).squeeze()
        y_test = pd.read_parquet(os.path.join(INPUT_PATH, f"y_test_{nombre_base}.parquet")).squeeze()

        # Entrenar modelo
        start_time = time.time()
        model = LogisticRegression(max_iter=500, multi_class='multinomial', solver='lbfgs')
        model.fit(X_train, y_train)
        elapsed_minutes = (time.time() - start_time) / 60

        # Predicciones
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)

        # Métricas
        report = classification_report(y_test, y_pred, output_dict=True)
        matrix = confusion_matrix(y_test, y_pred)
        recall_nivel_1 = recall_score(y_test == 1, y_pred == 1)
        f1_nivel_1 = f1_score(y_test == 1, y_pred == 1)

        if f1_nivel_1 > mejor_score:
            mejor_score = f1_nivel_1
            mejor_modelo = nombre_base

        # Guardar métricas CSV
        df_metrics = pd.DataFrame(report).transpose()
        df_metrics["elapsed_minutes"] = elapsed_minutes
        df_metrics["recall_nivel_1"] = recall_nivel_1
        df_metrics["f1_nivel_1"] = f1_nivel_1
        df_metrics.to_csv(os.path.join(OUTPUT_PATH, f"metricas_{nombre_base}.csv"))

        # Guardar visualizaciones PDF
        with PdfPages(os.path.join(OUTPUT_PATH, f"reporte_{nombre_base}.pdf")) as pdf:
            # Matriz de confusión
            ConfusionMatrixDisplay(matrix).plot(cmap='Blues')
            plt.title("Matriz de Confusión")
            pdf.savefig()
            plt.close()

            # Curvas ROC
            clases = np.unique(y_test)
            y_test_bin = label_binarize(y_test, classes=clases)
            plt.figure()
            for i, clase in enumerate(clases):
                fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
                plt.plot(fpr, tpr, label=f"Clase {clase} (AUC = {auc(fpr, tpr):.2f})")
            plt.plot([0, 1], [0, 1], "k--")
            plt.xlabel("False Positive Rate")
            plt.ylabel("True Positive Rate")
            plt.title("Curvas ROC por Clase")
            plt.legend()
            pdf.savefig()
            plt.close()

        # Agregar al resumen
        metricas_totales.append({
            "modelo": nombre_base,
            "f1_nivel_1": f1_nivel_1,
            "recall_nivel_1": recall_nivel_1,
            "tiempo_min": elapsed_minutes
        })

    except Exception as e:
        print(f"❌ Error en {nombre_base}: {e}")

# Guardar resumen general
df_resumen = pd.DataFrame(metricas_totales)
df_resumen.to_csv(os.path.join(OUTPUT_PATH, "resumen_modelos_logistic_regression.csv"), index=False)

print(f"\n🏆 Mejor modelo según F1 para nivel_triage=1: {mejor_modelo} (F1 = {mejor_score:.4f})")




🔍 Entrenando modelo: MaxAbs_FE_ALL


c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



🔍 Entrenando modelo: MaxAbs_FE_ANOVA


c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



🔍 Entrenando modelo: MaxAbs_FE_MI


...


🔍 Entrenando modelo: Standard_ORIGINAL_MI


c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



🔍 Entrenando modelo: Standard_ORIGINAL_VAR


c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



🏆 Mejor modelo según F1 para nivel_triage=1: MaxAbs_ORIGINAL_ALL (F1 = 0.1086)
